---
## **0. 준비**
---

In [1]:
%%html
<style>
    body {
        /* change --vscode-font-family if it's different on your system */
        --vscode-font-family: "SeoulNamsanC";
    }
</style>

In [1]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [3]:
import config
config.set_environment()

---
## **로컬 모델 실행**
---

- 로컬 모델의 장점:
    - 데이터에 대한 완전한 통제권과 개인정보 보호 보장
    - API 비용과 사용량 제한에서 자유로움
    - 인터넷 연결 없이 사용 가능
    - 모델 매개변수와 미세 조정을 직접 제어

---
### **Ollama 시작하기**
- Ollama는 개발자 친화 도구로 강력한 오픈 소스 모델을 로컬 환경에서 실행할 수 있도록 해줌
- 다양한 모델을 다운로드하고 실행할 수 있는 직관적인 인터페이스 제공

In [ ]:
# Ollama를 LCEL 패턴과 통합하는 방법 학습
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 선택한 모델로 Ollama 초기화
local_llm = ChatOllama(
    model = "qwen3.5:4B",
    base_url="http://172.30.128.1:11434",
    temperature=0,
)

# 로컬 모델을 사용하는 LECL 체인 생성
prompt = PromptTemplate.from_template("단순한 용어로 {concept}을 설명하라.")
local_chain = prompt | local_llm | StrOutputParser()

# 로컬 모델 체인 사용
result = local_chain.invoke({"concept": "양자 컴퓨팅"})
print(result)

---
### **로컬에서 Hugging Face 모델 작업**
- 허깅페이스는 모델 실행을 위한 두 가지 주요 방식을 지원
    - 로컬 환경에서 직접 실행하는 HuggingFacePipeline
    - 클라우드 기반 Hugging Face Hub를 통해 실행하는 HuggingFaceEndpoint
- 해당 실습에서는 로컬 실행 방식에 집중

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

# 소형 모델 파이프라인 생성
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation",
    pipeline_kwargs= dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
)

chat_model = ChatHuggingFace(llm=llm)

# 다른 LangChain LLM과 동일한 방식 사용
messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage(content= "양자 컴퓨팅에 대해 설명해주세요."),
]
ai_msg = chat_model.invoke(messages)
print(ai_msg.content)

---
### **로컬 모델을 위한 팁**
- 로컬 모델로 작업을 시작하면 성능 최적화와 일반적인 문제 해결이 필요.
- 다음 사항을 고려해야함.

<hr style="border:0;border-top:1.5px dashed gray;">

#### **1. 리소스 관리**
- 로컬 모델은 성능과 리소스 사용량을 균형있게 조절 필요
- 다음은 Ollama 모델을 효율적으로 운영하기 위한 구성 예시

In [ ]:
# 메모리, 연산 설정 최적화 모델 구성
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model= "qwen3.5:4b-q4_K_M",                                             #4비트 양자화 모델(메모리 사용량 감소)
    base_url="http://172.30.128.1:11434",                                                
    num_gpu=1,                                                              #사용할 GPU 개수(하드웨어에 따라 조정)
    num_thread=4,                                                           #병렬 처리용 CPUa 스레드 수 설정
    #temperature=0,                                                            
    #num_ctx=4096,                                                          #짧은 실습부터 시작해 메모리 사용량 제한               
)

- **`model="mistral:q4_K_M"`** : 4비트 양자화된 모델 지정, 양자화는 모델의 가중치를 더 적은 비트로 표현 -> 약간의 정확도는 손실되지만 메모리 크게 절약 가능
- **`num_gpu=1`** : GPU 리소스를 할당
    - 0 : CPU 전용 모드(GPU 없이 실행되나 속도 저하)
    - 1 : 단일 GPU 사용(일반 테스크톱 환경에 적합)
    - 더 높은 값 : 멀티 GPU 시스템용
- **`num_thread=4`** : CPU 병렬 처리를 제어
    - 낮은값(2-4) : 다른 프로그램과의 동시 실행에 적합
    - 높은값(8-16) : 전용 서버에서 최대 성능을 발휘
    - 최적값 : CPU의 물리적 코어 수에 맞추는 것이 좋음

In [ ]:
# 로컬 모델을 사용하는 LECL 체인 생성
prompt = PromptTemplate.from_template("단순한 용어로 {concept}을 설명하라.")
local_chain = prompt | llm | StrOutputParser()

# 로컬 모델 체인 사용
result = local_chain.invoke({"concept": "양자 컴퓨팅"})
print(result)

<hr style="border:0;border-top:1.5px dashed gray;">

#### **2. 오류 처리**
- 로컬 모델은 메모리 부족이나 예기치 않은 종료와 같은 다양한 오류가 발생할 수 있으므로 강력한 오류 처리 전략이 필수

In [ ]:
# 아래 코드 시간상 실제 실행-검증은 하지 않았음
import time

def safe_model_call(llm, prompt, max_retries=2):
    """재시도 로직과 안정적인 오류 대응을 갖춘 모델 안전 호출 함수"""
    retries = 0
    while retries <= max_retries:
        try:
            return llm.invoke(prompt)
        except RuntimeError as e:
            # VRAM 부족으로 로컬 모델 호출 중 자주 발생하는 오류
            if "CUDA out of memory in str(e)":
                print(f"GPU 메모리 에러, waiting and retrying ({retries+1}/{max_retries+1})")
                time.sleep(2) #시스템이 리소스를 해제할 시간을 줌
                retries += 1
            else:
                print(f"Runtime error: {e}")
                return "An error occured while processing your request."
        except Exception as e:
            print(f"Unexpected error calling model: {e}")
            return "An error occurred while processing your request."
        
        # 재시도 횟수를 모두 소진한 경우
        return "Model is currentrly experiencing high load. Please try again later."

# LECL chain에 안전 wrapper를 사용
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

prompt = PromptTemplate.from_template("단순한 용어로 {concept}을 설명하라.")
safe_llm = RunnableLambda(lambda x: safe_model_call(llm, x))
safe_chain = prompt | safe_llm
response = safe_chain.invoke({"concept": "양자 컴퓨팅"})
            

로컬 모델 사용 시 일반적으로 주요 오류는 다음과 같다.
- **메모리 부족** : 모델이 사용 가능한 VRAM을 초과하여 요구하는 경우
- **모델 로딩 실패** : 모델 파일이 손상되었거나 호환성 문제가 있는 경우
- **시간 초과** : 제한된 시스템 리소스로 인해 추론 시간이 지나치게 길어지는 경우
- **컨텍스트 길이 초과** : 입력이 모델의 최대 토근 제한을 넘어서는 경우


---
## **멀티모달 AI 애플리케이션(Skip)**
---